# Index · Unit 2 — Memory & state

Same shape as Unit 1. For every component in **Section 3 of `index.md`**, two cells:

- **From scratch** — the mechanism by hand, no framework. Runs with a scripted
  *fake model* / plain Python (no API key needed) so you see every moving part.
- **SOTA** — how it's actually done today (OpenAI SDK, embeddings, structured
  outputs). Needs `OPENAI_API_KEY`.

Components: **1** short-term / working memory · **2** long-term memory
(episodic / semantic / procedural) · **3** context management (summarization /
compaction) · **4** state management (scratchpad / progress).

**Intuition for the whole unit:** an LLM is *stateless* — it only knows what's
in the `messages` you send this call. "Memory" is everything you do *around*
the model to carry information forward: keep it in the window (short-term),
stash it in an external store and retrieve it (long-term), compress it when it
grows (context management), and track task progress in a structured object
(state).

## Setup

In [ ]:
import os, re, json, math
from collections import defaultdict

MODEL = "gpt-4o-mini"   # small, tool-capable model for the SOTA cells
EMBED = "text-embedding-3-small"

print("✅ key set — SOTA cells will run" if os.environ.get("OPENAI_API_KEY")
      else "⚠️  no OPENAI_API_KEY — from-scratch cells still run; SOTA cells need a key")

def chat(messages, model=MODEL, **kwargs):
    """Thin wrapper over the OpenAI SDK. Client built lazily so the
    from-scratch cells run without a key."""
    from openai import OpenAI
    return OpenAI().chat.completions.create(
        model=model, messages=messages, **kwargs
    ).choices[0].message

def embed(texts):
    """Embed a list of strings -> list of vectors."""
    from openai import OpenAI
    r = OpenAI().embeddings.create(model=EMBED, input=texts)
    return [d.embedding for d in r.data]

## 1. Short-term / working memory

The conversation history you pass on **every** call *is* the short-term memory.
It lives only in the `messages` list and is bounded by the **context window**.
Nothing outside that list exists for the model. So short-term memory = a
**buffer with a budget**: append turns, and when you exceed the budget, drop the
oldest (a sliding window).

**From scratch:** a bounded buffer that counts (approx) tokens and evicts the
oldest non-system messages when it overflows.

In [ ]:
# From scratch — a token-budgeted message buffer (sliding window).
def approx_tokens(msg): return max(1, len(msg["content"]) // 4)   # ~4 chars/token

class WorkingMemory:
    def __init__(self, budget=40):
        self.budget, self.msgs = budget, []
    def add(self, role, content):
        self.msgs.append({"role": role, "content": content})
        self._evict()
    def _evict(self):
        # keep system messages; drop oldest user/assistant turns until under budget
        while sum(approx_tokens(m) for m in self.msgs) > self.budget:
            i = next((k for k, m in enumerate(self.msgs) if m["role"] != "system"), None)
            if i is None: break
            dropped = self.msgs.pop(i)
            print(f"   ⟵ evicted: {dropped['role']}: {dropped['content'][:30]}...")

wm = WorkingMemory(budget=40)
wm.add("system", "You are a helpful assistant.")
for i in range(6):
    wm.add("user", f"This is message number {i} with some filler text.")
print("\nsurviving window:")
for m in wm.msgs: print(" ", m["role"], "->", m["content"][:40])

In [ ]:
# SOTA — the model only knows what's in `messages`. Prove it forgets evicted turns.
def recall_demo(history, question):
    return chat([*history, {"role": "user", "content": question}]).content

full = [{"role": "user", "content": "My name is Mia and my lucky number is 7."},
        {"role": "assistant", "content": "Nice to meet you, Mia!"}]
print("with history :", recall_demo(full, "What is my lucky number?"))
print("window dropped:", recall_demo([], "What is my lucky number?"))
# Same model, same question — the only difference is what survived in the window.

## 2. Long-term memory — episodic / semantic / procedural

Short-term memory dies when the window slides. **Long-term memory** persists in
an *external store* across sessions, and you **retrieve** the relevant bits back
into the window when needed. Three classic kinds:

- **episodic** — past events / what happened ("on 2026-09-01 the user asked X").
- **semantic** — facts about the world or user ("the user's name is Mia").
- **procedural** — how-to / skills ("to deploy, run `make ship`").

**From scratch:** a dict with three namespaces + keyword retrieval.

In [ ]:
# From scratch — a 3-namespace store with naive keyword retrieval.
class LongTermMemory:
    def __init__(self): self.store = {"episodic": [], "semantic": [], "procedural": []}
    def write(self, kind, text): self.store[kind].append(text)
    def recall(self, query, k=2):
        q = set(query.lower().split())
        scored = [(len(q & set(t.lower().split())), t)
                  for items in self.store.values() for t in items]
        return [t for s, t in sorted(scored, reverse=True) if s > 0][:k]

ltm = LongTermMemory()
ltm.write("semantic",   "The user's name is Mia.")
ltm.write("semantic",   "Mia's lucky number is 7.")
ltm.write("episodic",   "On 2026-09-01 Mia asked about agent memory.")
ltm.write("procedural", "To reset memory, call ltm.store.clear().")

print("recall 'what is mia lucky number' ->", ltm.recall("what is mia lucky number"))
# Then you'd inject these strings into the system prompt before calling the model.

In [ ]:
# SOTA — semantic memory via embeddings + cosine similarity (real vector recall).
def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na, nb = math.sqrt(sum(x*x for x in a)), math.sqrt(sum(y*y for y in b))
    return dot / (na*nb + 1e-9)

class VectorMemory:
    def __init__(self): self.texts, self.vecs = [], []
    def write(self, texts):
        self.texts += texts
        self.vecs  += embed(texts)
    def recall(self, query, k=2):
        qv = embed([query])[0]
        ranked = sorted(self.texts, key=lambda t: cosine(qv, self.vecs[self.texts.index(t)]),
                        reverse=True)
        return ranked[:k]

vm = VectorMemory()
vm.write(["The user's name is Mia.", "Mia's lucky number is 7.",
          "The capital of France is Paris.", "Mia prefers dark mode."])
# Note "digit she likes" shares NO keywords — embeddings still match on meaning.
print(vm.recall("which digit does she like?"))
print(vm.recall("ui theme preference?"))

## 3. Context management / engineering — summarization & compaction

The window is finite, so you can't just keep appending forever. **Context
management** decides what to keep verbatim, what to compress, and what to drop.
The workhorse is **compaction**: replace a batch of old turns with a short
summary, keep the last few turns intact.

**From scratch:** when the buffer exceeds a limit, compress all-but-the-last-N
turns into one crude extractive summary message.

In [ ]:
# From scratch — compaction via a crude extractive summary.
def compact(msgs, keep_last=2):
    if len(msgs) <= keep_last + 1: return msgs
    old, recent = msgs[:-keep_last], msgs[-keep_last:]
    # extractive "summary": first sentence of each old turn, joined
    facts = "; ".join(m["content"].split(".")[0] for m in old if m["role"] != "system")
    summary = {"role": "system", "content": f"[summary of earlier turns] {facts}"}
    return [summary, *recent]

convo = [
    {"role": "user", "content": "I'm planning a trip to Japan. My budget is $3000."},
    {"role": "assistant", "content": "Great. When are you traveling?"},
    {"role": "user", "content": "In April. I want to see Kyoto and Tokyo."},
    {"role": "assistant", "content": "April is cherry-blossom season."},
    {"role": "user", "content": "What should I pack?"},
]
for m in compact(convo): print(m["role"], "->", m["content"][:60])

In [ ]:
# SOTA — let the model write the summary (real compaction, like Claude Code's /compact).
def llm_compact(msgs, keep_last=2):
    if len(msgs) <= keep_last + 1: return msgs
    old, recent = msgs[:-keep_last], msgs[-keep_last:]
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in old)
    s = chat([
        {"role": "system", "content": "Summarize this conversation into 1-2 sentences "
                                       "capturing durable facts/decisions. Be terse."},
        {"role": "user", "content": transcript},
    ]).content
    return [{"role": "system", "content": f"[summary] {s}"}, *recent]

for m in llm_compact(convo): print(m["role"], "->", m["content"][:80])

## 4. State management — scratchpad & progress

Chat history is lossy and unstructured. For multi-step tasks the agent also
keeps an explicit **state object** — a scratchpad separate from the messages —
tracking todo / done / intermediate results. The agent reads it to decide the
next step and writes to it after each step.

**From scratch:** a plain dict with `todo` / `done` / `scratch`, mutated by
helper functions.

In [ ]:
# From scratch — an explicit task-state scratchpad.
state = {"goal": "Make tea", "todo": ["boil water", "add tea", "pour"],
         "done": [], "scratch": {}}

def step(state):
    if not state["todo"]: return False
    task = state["todo"].pop(0)
    state["scratch"]["last_action"] = task
    state["done"].append(task)
    print(f"did: {task:12}  | remaining: {state['todo']}")
    return True

while step(state): pass
print("\nfinal state:", json.dumps(state, indent=2))

In [ ]:
# SOTA — structured outputs keep the state typed; the model proposes the next update.
def next_state(state):
    schema = {"type": "object", "properties": {
        "next_action": {"type": "string"},
        "remaining":   {"type": "array", "items": {"type": "string"}},
        "done":        {"type": "boolean"}}, "required": ["next_action","remaining","done"],
        "additionalProperties": False}
    m = chat([
        {"role": "system", "content": "You advance a task. Given the state, output the next "
                                      "action and the remaining todo list as JSON."},
        {"role": "user", "content": json.dumps(state)},
    ], response_format={"type":"json_schema","json_schema":{"name":"state","schema":schema,"strict":True}})
    return json.loads(m.content)

print(next_state({"goal":"Make tea","todo":["boil water","add tea","pour"],"done":[]}))
# No regex, no parsing by hand — the model is constrained to emit valid typed state.

## Recap

Memory is everything you build **around** a stateless model to carry information
forward:

| component | mechanism (from scratch) | SOTA packaging |
|---|---|---|
| short-term | token-budgeted sliding-window buffer | the `messages` list you send each call |
| long-term | 3-namespace store + keyword recall | embeddings + vector similarity recall |
| context mgmt | extractive summary of old turns | model-written compaction (`/compact`) |
| state | a `todo/done/scratch` dict | structured outputs → typed state |

The through-line of Unit 1 holds: each piece is a few lines of plain
Python (a buffer / a store / a summary / a dict), and the SOTA version is that
same idea packaged with native model features (embeddings, structured outputs).
Frameworks (LangGraph, LlamaIndex, mem0) just ship these for you.

Next in `index.md`: **Unit 3 — Knowledge & grounding (RAG)**, which builds
directly on the vector-memory cell above.